# ISRU system connectivity

Replacement for Fig. 2 of the SciTech V1 manuscript. The selected case includes seven system types. The figure uses the interfaces between those types in the archived ISRU graph. Arrows indicate physical flow direction. Run both cells from this checkout to export the vector PDF and PNG beside the notebook.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyArrowPatch
from matplotlib.lines import Line2D

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "outputs/scenarios/ISRU_Mission/graph.json").is_file())
graph = json.loads((root / "outputs/scenarios/ISRU_Mission/graph.json").read_text())
destination = root / "paper_scitech/figures/isru_system_connectivity"
# One node per system type. Separate rover instances belong in the operations diagram.
nodes = {
    "ISRUExcavation": ("ISRU\nExcavation", 0.71, 0.52),
    "RegolithRover": ("Regolith\nRover", 2.21, 0.52),
    "ISRUPlant": ("ISRU\nPlant", 3.71, 0.52),
    "LOXRover": ("LOX\nRover", 5.21, 0.52),
    "PropellantDepot": ("Propellant\nDepot", 6.71, 0.52),
    "SolarPowerSystem": ("Solar Power\nSystem", 3.71, 1.96),
    "LaunchLandingZone": ("Launch /\nLanding Zone", 6.71, 1.96),
}
assert set(nodes) <= set(graph["active_nodes"])
edges = []
def port_directions(endpoint):
    ports = graph["parts"][endpoint["part"]]["ports"]
    return set(next(p["direction"] for p in ports if p["name"] == endpoint["port"]))

for link in graph["connections"]:
    a, b = link["from"], link["to"]
    if a["part"] not in nodes or b["part"] not in nodes:
        continue
    # Orient by the source/output and receiver/input ports. An interface may
    # list a bidirectional rover port first even when the plant is supplying it.
    da, db = port_directions(a), port_directions(b)
    forward = "out" in da and "in" in db
    reverse = "out" in db and "in" in da
    if forward == reverse:
        raise ValueError(f"Ambiguous flow direction: {link['name']}")
    src, dst = (a, b) if forward else (b, a)
    edges.append((src["part"], dst["part"], link["flow"]))

from collections import Counter
assert Counter(flow for _, _, flow in edges) == {"Power": 4, "LOX": 2, "Regolith": 2}


In [ ]:
styles = {
    "Power": ("#AE323D", "solid"),
    "LOX": ("#176597", "solid"),
    "Regolith": ("#815338", "solid"),
}
plt.rcParams.update({"pdf.fonttype": 42, "font.family": "DejaVu Sans"})
fig, ax = plt.subplots(figsize=(7.45, 2.50))
fig.subplots_adjust(left=0.005, right=0.995, bottom=0.02, top=0.98)
ax.set(xlim=(0, 7.42), ylim=(0, 2.47), aspect="equal")
ax.axis("off")
boxes = {}
for system, (label, x, y) in nodes.items():
    box = Ellipse((x, y), 1.13, 0.65, facecolor="#DCF0F3",
                  edgecolor="#24383D", linewidth=0.95, zorder=2)
    ax.add_patch(box)
    boxes[system] = box
    ax.text(x, y, label, ha="center", va="center", fontsize=9.5,
            linespacing=1.12, color="#16282E", zorder=3)

for source, target, flow in edges:
    color, style = styles[flow]
    ax.add_patch(FancyArrowPatch(
        boxes[source].center, boxes[target].center,
        patchA=boxes[source], patchB=boxes[target], shrinkA=1, shrinkB=1.2,
        arrowstyle="-|>", mutation_scale=10.5, linewidth=1.55,
        linestyle=style, color=color, zorder=1))

handles = [Line2D([], [], color=color, linestyle=style, linewidth=1.8, label=flow)
           for flow, (color, style) in styles.items()]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.022, 0.985),
          frameon=False, handlelength=2.2, labelspacing=0.38, fontsize=8.6)
fig.savefig(destination.with_suffix(".pdf"), facecolor="white", bbox_inches="tight", pad_inches=0.025)
fig.savefig(destination.with_suffix(".png"), dpi=320, facecolor="white", bbox_inches="tight", pad_inches=0.025)
plt.show()
